In [1]:
import os
%pwd
os.chdir("../")
%pwd

'd:\\Data Science\\END to END Proj\\DrugToxicity'

In [2]:
## ENTITY
from dataclasses import dataclass
from pathlib import Path
from typing import Dict

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    STATUS_FILE: str
    data_path: Path

In [3]:
from src.DrugToxicity.constant import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from src.DrugToxicity.utils.common import read_yaml, create_directories

In [4]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation

        create_directories([config.root_dir])

        return DataValidationConfig(
            root_dir=Path(config.root_dir),
            STATUS_FILE=config.STATUS_FILE,
            data_path=Path(config.data_path)
        )

In [5]:
import pandas as pd
from pathlib import Path
from src.DrugToxicity.utils.common import logger

## Expected Tox21 schema — columns and their pandas dtypes
TOX21_SCHEMA = {
    "smiles"       : "object",
    "mol_id"       : "object",
    "NR-AR"        : "float64",
    "NR-AR-LBD"    : "float64",
    "NR-AhR"       : "float64",
    "NR-Aromatase" : "float64",
    "NR-ER"        : "float64",
    "NR-ER-LBD"    : "float64",
    "NR-PPAR-gamma": "float64",
    "SR-ARE"       : "float64",
    "SR-ATAD5"     : "float64",
    "SR-HSE"       : "float64",
    "SR-MMP"       : "float64",
    "SR-p53"       : "float64",
}

In [6]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    def validate_dataset(self) -> bool:
        """
        Validates the Tox21 CSV:
          1. Skips if STATUS_FILE already exists
          2. All required columns present
          3. Column dtypes match schema
          4. smiles column has at least some non-null values
          5. At least one target column has labelled rows
          6. Minimum row count sanity check
        Writes result to STATUS_FILE.
        Returns True if valid, raises ValueError if not.
        """
        status_path = Path(self.config.STATUS_FILE)

        ## Skip if already validated
        if status_path.exists():
            logger.info(
                f"Validation status file already exists at {status_path} — skipping validation."
            )
            with open(status_path) as f:
                return "True" in f.read()

        validation_status = True
        issues = []

        try:
            df = pd.read_csv(self.config.data_path)

            ## 1. Required columns present
            missing = set(TOX21_SCHEMA.keys()) - set(df.columns)
            if missing:
                issues.append(f"Missing columns: {missing}")
                validation_status = False

            ## 2. Data types
            for col, expected_type in TOX21_SCHEMA.items():
                if col not in df.columns:
                    continue
                actual = str(df[col].dtype)
                if expected_type == "float64" and "float" not in actual:
                    issues.append(f"Type mismatch '{col}': expected float64, got {actual}")
                    validation_status = False
                if expected_type == "object" and actual != "object":
                    issues.append(f"Type mismatch '{col}': expected object, got {actual}")
                    validation_status = False

            ## 3. smiles not all null
            if "smiles" in df.columns and df["smiles"].isna().all():
                issues.append("All SMILES values are null")
                validation_status = False

            ## 4. At least one target has labels
            target_cols = [c for c in TOX21_SCHEMA if c not in ("smiles", "mol_id")]
            present = [c for c in target_cols if c in df.columns]
            if present and df[present].notna().sum().max() == 0:
                issues.append("All target columns are entirely null")
                validation_status = False

            ## 5. Minimum row count
            if len(df) < 100:
                issues.append(f"Dataset too small: only {len(df)} rows")
                validation_status = False

            logger.info(f"Dataset loaded: {len(df)} rows, {len(df.columns)} columns")

        except Exception as e:
            issues.append(f"Exception during validation: {e}")
            validation_status = False

        ## Write status file
        self.config.root_dir.mkdir(parents=True, exist_ok=True)
        with open(status_path, "w") as f:
            f.write(f"Validation status: {validation_status}\n")
            for issue in issues:
                f.write(f"  - {issue}\n")

        if validation_status:
            logger.info("Data validation PASSED.")
        else:
            logger.error(f"Data validation FAILED. Issues: {issues}")

        return validation_status

In [7]:
try:
    config_manager = ConfigurationManager()
    data_validation_config = config_manager.get_data_validation_config()

    validator = DataValidation(config=data_validation_config)
    is_valid  = validator.validate_dataset()

    if not is_valid:
        raise ValueError(
            "Data validation failed — check "
            f"{data_validation_config.STATUS_FILE} for details."
        )

    print("Data validation passed successfully")

except Exception as e:
    print(f"Error during data validation: {str(e)}")
    raise e

[2026-03-27 00:03:03,833: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-27 00:03:03,837: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-27 00:03:03,839: INFO: common: created directory at: artifacts]
[2026-03-27 00:03:03,842: INFO: common: created directory at: artifacts/data_validation]
[2026-03-27 00:03:03,904: INFO: 3077872406: Dataset loaded: 7831 rows, 14 columns]
[2026-03-27 00:03:03,904: INFO: 3077872406: Data validation PASSED.]
Data validation passed successfully
